**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 02 - Indicator Extraction

Extracts the remaining RBI State indicators using the parser in src/parser.py,
cleans them, and saves each as a tidy CSV in data/processed.

## Setup

In [1]:
import sys
from pathlib import Path

# find the project root (the folder that contains 'src')
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
from src.parser import locate_table, parse_table

text = (root / "data" / "processed" / "rbi_states_handbook_2024_25.txt").read_text(encoding="utf-8")
life = parse_table(locate_table(text, 7))
print(life.shape, "|", life["state"].nunique(), "states")
print(life.head())

(491, 3) | 22 states
            state     year  value
0  Andhra Pradesh  1993-97   61.2
1  Andhra Pradesh  1994-98   63.5
2  Andhra Pradesh  1995-99   62.4
3           Assam  1993-97   56.6
4           Assam  1994-98   57.1


## Batch extraction

In [2]:
tables = {
    "life_expectancy": 7,
    "per_capita_power": 138,
    "td_losses": 148,
    "roads": 146,
    "telephones": 149,
    "manufacturing_gva": 33,
    "total_gva": 25,
    "per_capita_nsdp": 19,
    "total_nsdp": 23,
}

out_dir = root / "data" / "processed"
for name, tnum in tables.items():
    df = parse_table(locate_table(text, tnum))
    df.to_csv(out_dir / f"{name}.csv", index=False)
    print(f"{name:18} T{tnum:<4} rows={df.shape[0]:<5} states={df['state'].nunique()}")

life_expectancy    T7    rows=491   states=22
per_capita_power   T138  rows=745   states=36
td_losses          T148  rows=683   states=37
roads              T146  rows=576   states=36
telephones         T149  rows=594   states=28
manufacturing_gva  T33   rows=470   states=34
total_gva          T25   rows=470   states=34
per_capita_nsdp    T19   rows=470   states=34
total_nsdp         T23   rows=470   states=34


## Unemployment extraction

In [3]:
def parse_subblock(text, table_number, label):
    block = locate_table(text, table_number)
    start = block.find(label)
    if start == -1:
        raise ValueError(f"label '{label}' not found in table {table_number}")
    return parse_table(block[start:])

rural = parse_subblock(text, 8, "(Rural Overall)")
urban = parse_subblock(text, 9, "(Urban Overall)")

rural.to_csv(out_dir / "unemployment_rural_overall.csv", index=False)
urban.to_csv(out_dir / "unemployment_urban_overall.csv", index=False)

print("rural:", rural.shape, rural["state"].nunique(), "states")
print("urban:", urban.shape, urban["state"].nunique(), "states")
print(rural[rural["state"] == "Bihar"].tail(3))

rural: (432, 3) 36 states
urban: (432, 3) 36 states
    state     year  value
45  Bihar  2021-22   55.0
46  Bihar  2022-23   36.0
47  Bihar  2023-24   26.0


## Credit-deposit extraction

In [4]:
cd_util = parse_table(locate_table(text, 154))          # place of utilisation (primary)
cd_util.to_csv(out_dir / "cd_ratio_utilisation.csv", index=False)

cd_sanc = parse_table(locate_table(text, 153))          # place of sanction (reference)
cd_sanc.to_csv(out_dir / "cd_ratio_sanction.csv", index=False)

print("utilisation:", cd_util.shape, cd_util["state"].nunique(), "states")
print("sanction:   ", cd_sanc.shape, cd_sanc["state"].nunique(), "states")
print(cd_util[cd_util["state"] == "Bihar"].tail(3))

utilisation: (814, 3) 38 states
sanction:    (814, 3) 38 states
     state  year  value
591  Bihar  2023   48.5
592  Bihar  2024   52.8
593  Bihar  2025   53.5


## MSME extraction

In [5]:
import re
import pdfplumber
from src.parser import is_value, clean_number

msme_pdf = next(p for p in (root / "data" / "raw").iterdir()
                if "msme" in p.name.lower() and p.suffix.lower() == ".pdf")
with pdfplumber.open(msme_pdf) as pdf:
    msme_text = "\n".join(p.extract_text() or "" for p in pdf.pages)

# rows that fit on one line: SNo State Micro Small Medium Total, where Total = the sum
records = []
for line in msme_text.splitlines():
    toks = line.split()
    if len(toks) < 6 or not re.fullmatch(r"\d{1,2}", toks[0]):
        continue
    vals = toks[-4:]
    if not all(is_value(v) for v in vals):
        continue
    state = " ".join(toks[1:-4])
    if not state[:1].isalpha():
        continue
    micro, small, medium, total = (clean_number(v) for v in vals)
    if None in (micro, small, medium, total):
        continue
    if abs(total - (micro + small + medium)) <= 1:
        records.append((state, micro, small, medium, total))

# three UTs whose names wrap across lines (or have missing sub-values) are added
# from the source table after manual verification (see DATA_DECISIONS.md)
manual = [
    ("Andaman & Nicobar Islands", 21645, 156, 4, 21805),
    ("Dadra & Nagar Haveli and Daman & Diu", 33931, 916, 135, 34982),
    ("Lakshadweep", 2431, None, None, 2431),
]
records += manual

udyam = (pd.DataFrame(records, columns=["state", "micro", "small", "medium", "total"])
         .drop_duplicates("state").reset_index(drop=True))
udyam.to_csv(out_dir / "msme_udyam.csv", index=False)
print("udyam:", udyam.shape, "states:", udyam["state"].nunique())

udyam: (36, 5) states: 36


## Summary

In [6]:
processed = sorted((root / "data" / "processed").glob("*.csv"))
print(f"{len(processed)} tables saved in data/processed:\n")
for p in processed:
    df = pd.read_csv(p)
    print(f"  {p.name:32} rows={len(df):<5} states={df['state'].nunique() if 'state' in df.columns else '-'}")

17 tables saved in data/processed:

  cd_ratio_sanction.csv            rows=814   states=38
  cd_ratio_utilisation.csv         rows=814   states=38
  factories_t116.csv               rows=740   states=37
  ger_t1.csv                       rows=1296  states=36
  gfcf_t130.csv                    rows=740   states=37
  life_expectancy.csv              rows=491   states=22
  manufacturing_gva.csv            rows=470   states=34
  msme_udyam.csv                   rows=36    states=36
  per_capita_nsdp.csv              rows=470   states=34
  per_capita_power.csv             rows=745   states=36
  roads.csv                        rows=576   states=36
  td_losses.csv                    rows=683   states=37
  telephones.csv                   rows=594   states=28
  total_gva.csv                    rows=470   states=34
  total_nsdp.csv                   rows=470   states=34
  unemployment_rural_overall.csv   rows=432   states=36
  unemployment_urban_overall.csv   rows=432   states=36
